# Plot R7378A JSON Data

This notebook loads `DATA/R7378A_Characteristics.json` and `DATA/R7378A_Spectral_Response.json` from the King-CRAB-Analysis folder and plots them as separate figures.


In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# Local JSON files in the project DATA folder.
# This works whether the notebook kernel starts in King-CRAB-Analysis
# or in King-CRAB-Analysis/CODE.
project_folder = Path.cwd()
if not (project_folder / "DATA").exists() and (project_folder.parent / "DATA").exists():
    project_folder = project_folder.parent

data_folder = project_folder / "DATA"
characteristics_json = data_folder / "R7378A_Characteristics.json"
spectral_response_json = data_folder / "R7378A_Spectral_Response.json"

for path in [characteristics_json, spectral_response_json]:
    if not path.exists():
        raise FileNotFoundError(f"Could not find {path}")
    print(f"Found: {path}")


In [ ]:
def load_webplotdigitizer_json(json_path):
    with json_path.open("r") as f:
        data = json.load(f)

    datasets = []
    for dataset in data.get("datasetColl", []):
        name = dataset.get("name", "Dataset")
        xy = []

        for point in dataset.get("data", []):
            # WebPlotDigitizer stores calibrated coordinates in value = [x, y].
            if "value" in point and len(point["value"]) >= 2:
                xy.append(point["value"][:2])

        if xy:
            xy = np.array(xy, dtype=float)
            xy = xy[np.argsort(xy[:, 0])]
            datasets.append({"name": name, "x": xy[:, 0], "y": xy[:, 1]})

    return datasets


characteristics = load_webplotdigitizer_json(characteristics_json)
spectral_response = load_webplotdigitizer_json(spectral_response_json)

for label, datasets in [("Characteristics", characteristics), ("Spectral Response", spectral_response)]:
    print(f"\n{label}")
    for dataset in datasets:
        x = dataset["x"]
        y = dataset["y"]
        print(f"  {dataset['name']}: {len(x)} points")
        print(f"    x range: {np.nanmin(x):.4g} to {np.nanmax(x):.4g}")
        print(f"    y range: {np.nanmin(y):.4g} to {np.nanmax(y):.4g}")


In [ ]:
def plot_two_axis_datasets(datasets, title, left_label, right_label=None, yscale="linear", xlim=None, ylim=None):
    fig, ax_left = plt.subplots(figsize=(8, 5))
    colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

    if len(datasets) == 0:
        raise ValueError(f"No datasets to plot for {title}")

    first = datasets[0]
    ax_left.plot(first["x"], first["y"], "o-", ms=4, lw=1.8, color=colors[0], label=first["name"])
    ax_left.set_xlabel("Wavelength / Voltage")
    ax_left.set_ylabel(left_label, color=colors[0])
    ax_left.tick_params(axis="y", labelcolor=colors[0])
    ax_left.grid(True, alpha=0.3, which="both")
    if yscale == "log":
        ax_left.set_yscale("log")
    if xlim is not None:
        ax_left.set_xlim(*xlim)
    if ylim is not None:
        ax_left.set_ylim(*ylim)

    axes = [ax_left]
    lines = ax_left.get_lines()

    if len(datasets) > 1:
        second = datasets[1]
        ax_right = ax_left.twinx()
        ax_right.plot(second["x"], second["y"], "s-", ms=4, lw=1.8, color=colors[1], label=second["name"])
        ax_right.set_ylabel(right_label or second["name"], color=colors[1])
        ax_right.tick_params(axis="y", labelcolor=colors[1])
        if yscale == "log":
            ax_right.set_yscale("log")
        if ylim is not None:
            ax_right.set_ylim(*ylim)
        axes.append(ax_right)
        lines += ax_right.get_lines()

    for extra, color in zip(datasets[2:], colors[2:]):
        ax_left.plot(extra["x"], extra["y"], "o-", ms=4, lw=1.5, color=color, label=extra["name"])
        lines += ax_left.get_lines()[-1:]

    labels = [line.get_label() for line in lines]
    ax_left.legend(lines, labels, loc="best")
    ax_left.set_title(title)
    fig.tight_layout()
    plt.show()


In [ ]:
# Plot 1: R7378A characteristics.
# Gain and anode dark current use very different y-scales, so they are shown on separate y-axes.
plot_two_axis_datasets(
    characteristics,
    title="Typical Gain and Dark Current Charactersitics",
    left_label="Gain",
    right_label="Anode Dark Current A",
    yscale="log",
)


In [ ]:
# Plot 2: R7378A spectral response.
# Quantum efficiency and cathode radiant sensitivity are separate quantities, so they are shown on separate y-axes.
plot_two_axis_datasets(
    spectral_response,
    title="Typical Spectral Response",
    left_label="Quantum Efficiency [%]",
    right_label="Cathode Radiant Sensitivity (mA/W)",
    yscale="log",
    xlim=(100, 800),
    ylim=(0.01, 100),
)
